In [15]:
import os

In [35]:
import warnings
warnings.filterwarnings("ignore")

In [36]:
from dotenv import load_dotenv
load_dotenv()

True

In [37]:

google_api_key = os.getenv("GOOGLE_API_KEY")
langchain_api_key = os.getenv("LANGCHAIN_API_KEY")
langsmith_tracing = os.getenv("LANGSMITH_TRACING")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
# langsmith_endpoint = os.getenv("LANGSMITH_ENDPOINT")
# langsmith_project = os.getenv("LANGSMITH_PROJECT")



# LOAD THE MODEL AND TEST IT WITH SIMPLE MESSGAE

In [19]:
from langchain_core.messages import HumanMessage, SystemMessage


In [20]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [21]:
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", convert_system_message_to_human=True, temperature=0)

In [22]:
message = [SystemMessage(content="Hi you are nice bot"), HumanMessage(content="Hi, how are you buddy")]

In [23]:
response = model.invoke(message)

# Use output parser

In [24]:
from langchain_core.output_parsers import StrOutputParser
parser  = StrOutputParser()

In [25]:
# parser.invoke(response)

# chaining with LCEL

In [26]:
chain  = model | parser

In [27]:
chain.invoke("hello what is your name")
chain.invoke("what is bard")

'Bard is a large language model (LLM) chatbot developed by Google AI. It\'s designed to be conversational and informative, able to generate text, translate languages, write different kinds of creative content, and answer your questions in an informative way.\n\nHere\'s a breakdown of what that means:\n\n*   **Large Language Model (LLM):** This means Bard is trained on a massive amount of text data, allowing it to understand and generate human-like text.\n*   **Chatbot:** It\'s designed to interact with users in a conversational manner, responding to prompts and questions.\n*   **Developed by Google AI:** This indicates that Bard is a product of Google\'s artificial intelligence research and development efforts.\n\n**Key Capabilities:**\n\n*   **Text Generation:** Bard can create various types of text, such as articles, poems, code, scripts, musical pieces, email, letters, etc.\n*   **Translation:** It can translate text between multiple languages.\n*   **Question Answering:** Bard can 

# prompting with llm

In [28]:
from langchain_core.prompts import ChatPromptTemplate

In [29]:



prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Translate the following into {language}:"),
    ("user", "{text}"),
])

In [30]:

prompt = prompt_template.invoke({"language" : "japanse", "text" : "hello my name is sheryar"})
prompt.to_messages

<bound method ChatPromptValue.to_messages of ChatPromptValue(messages=[SystemMessage(content='Translate the following into japanse:', additional_kwargs={}, response_metadata={}), HumanMessage(content='hello my name is sheryar', additional_kwargs={}, response_metadata={})])>

In [31]:
chain = prompt_template | model | parser
result = chain.invoke({"language" : "japanse", "text" : "hello my name is sheryar"})

In [32]:
result

'Here are a few ways to translate "Hello, my name is Sheryar" into Japanese, with slight variations in formality:\n\n*   **こんにちは、私の名前はシェリヤールです。** (Konnichiwa, watashi no namae wa Sheriyar desu.) - This is a polite and standard translation. "Konnichiwa" means "Hello," "watashi no namae wa" means "my name is," and "Sheriyar desu" means "is Sheryar."\n\n*   **こんにちは、シェリヤールと申します。** (Konnichiwa, Sheriyar to mōshimasu.) - This is a more humble and polite way to introduce yourself. "to mōshimasu" is a humble way of saying "my name is."\n\n*   **こんにちは、シェリヤールです。** (Konnichiwa, Sheriyar desu.) - This is a slightly more casual but still polite option. It omits "my name is" but is perfectly acceptable.\n\n*   **どうも、シェリヤールです。** (Dōmo, Sheriyar desu.) - "Dōmo" is a versatile greeting that can mean "Hello," "Hi," or "Thanks." This is a bit more informal than "Konnichiwa."\n\nThe best option depends on the context and who you are speaking to. If you want to be safe and polite, the first option is a goo

In [33]:
model.invoke(prompt)

AIMessage(content='Here are a few ways to translate "Hello, my name is Sheryar" into Japanese, with slight variations in formality:\n\n*   **こんにちは、私の名前はシェリヤールです。** (Konnichiwa, watashi no namae wa Sheriyar desu.) - This is a polite and standard translation. "Konnichiwa" means "Hello," "watashi no namae wa" means "my name is," and "Sheriyar desu" means "is Sheryar."\n\n*   **こんにちは、シェリヤールと申します。** (Konnichiwa, Sheriyar to mōshimasu.) - This is a more humble and polite way to introduce yourself. "to mōshimasu" is a humble way of saying "my name is."\n\n*   **こんにちは、シェリヤールです。** (Konnichiwa, Sheriyar desu.) - This is a slightly more casual but still polite option. It omits "my name is" but is perfectly acceptable.\n\n*   **どうも、シェリヤールです。** (Dōmo, Sheriyar desu.) - "Dōmo" is a versatile greeting that can mean "Hello," "Hi," or "Thanks." This is a bit more informal than "Konnichiwa."\n\nThe best option depends on the context and who you are speaking to. If you want to be safe and polite, the fir

In [34]:
chain.invoke({"language" : "japanse", "text" : "hungry"})

'The most common and direct translation of "hungry" into Japanese is:\n\n*   **お腹が空いた (Onaka ga suita)** - This literally means "My stomach is empty" and is the most natural way to express hunger.\n\nHere are a few other options, depending on the nuance you want to convey:\n\n*   **空腹 (Kūfuku)** - This is a more formal and slightly literary word for "hunger."\n*   **ぺこぺこ (Peko peko)** - This is an onomatopoeia (sound-symbolic word) that describes the feeling of being very hungry. It\'s more casual and cute. You can say "お腹がぺこぺこだ (Onaka ga peko peko da)" which means "My stomach is rumbling (with hunger)."\n*   **お腹すいた (Onaka suita)** - This is a more casual version of "お腹が空いた (Onaka ga suita)".\n\nSo, the best translation depends on the context, but **お腹が空いた (Onaka ga suita)** is generally the safest and most common choice.'

# agents


In [3]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

In [5]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage

In [39]:
memory = MemorySaver()
search = TavilySearchResults(max_results=2)

In [40]:
tools = [search]

In [43]:
agent_executor = create_react_agent(model,tools, checkpointer=memory)

In [44]:
config = {"configurable" : {"thread_id"   : "abc123"}}

In [46]:
for chunk in agent_executor.stream({"messages" : [HumanMessage(content="hi my name is sheryar i live in gilgit baltistan")]}, config):
    print(chunk)
    print("---------------")

{'agent': {'messages': [AIMessage(content="Hello Sheryar from Gilgit Baltistan! It's nice to meet you.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--82c4a94b-6f35-4a01-9403-9a0c72545c38-0', usage_metadata={'input_tokens': 64, 'output_tokens': 20, 'total_tokens': 84, 'input_token_details': {'cache_read': 0}})]}}
---------------


In [47]:
for chunk in agent_executor.stream({"messages" : [HumanMessage(content="whats the weather where i live?")]}, config):
    print(chunk)
    print("---------------")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_results_json', 'arguments': '{"query": "weather in Gilgit Baltistan"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--c96f976a-7761-40c6-8ed2-f4ff333f880c-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Gilgit Baltistan'}, 'id': 'c4fdf447-27e6-4ccd-be7c-2749f939b365', 'type': 'tool_call'}], usage_metadata={'input_tokens': 90, 'output_tokens': 16, 'total_tokens': 106, 'input_token_details': {'cache_read': 0}})]}}
---------------
{'tools': {'messages': [ToolMessage(content='[{"title": "Weekly Weather Outlook (English)", "url": "https://nwfc.pmd.gov.pk/new/weekly-outlook-en.php", "content": "| 19 September, 2025Friday | Rain-windstorm/thundershower is likely in Upper Khyber Pakhtunkhwa, Islamabad, Pothohar region, northeas